# Taller: construye y ajusta tu propio clasificador de imágenes

Este notebook documenta un problema de clasificación de imágenes construido con un dataset propio. Se comparan dos modelos basados en transferencia de aprendizaje:

1. Una base convolucional preentrenada congelada.
2. La misma base con fine-tuning de sus últimas capas.

> Antes de ejecutar: crea la carpeta `datos_imagenes_originales/` y una subcarpeta por clase. Por ejemplo: `datos_imagenes_originales/clase_a/imagen_01.jpg`. Usa imágenes propias y evita incluir la misma imagen en más de una partición.

La partición de test se reserva para la evaluación final y no se usa durante el entrenamiento inicial.

## 1. Definición del problema y estrategia de búsqueda

**Problema:** clasificar una imagen en una de las clases definidas a partir de fotografías recolectadas por el estudiante.

**Pregunta:** ¿un modelo preentrenado en ImageNet puede distinguir las clases de este dataset propio y mejora su desempeño al descongelar parcialmente la base convolucional?

**Estrategia de búsqueda de datos:** buscaré imágenes representativas de cada clase en fuentes permitidas por el curso, procurando variedad de iluminación, fondo, distancia y orientación. Eliminaré duplicados, imágenes borrosas y fotografías que no correspondan a la clase. La partición será estratificada y se mantendrá fija para que la comparación antes/después sea justa.

**Criterio de éxito:** además de la exactitud, revisaré `precision`, `recall`, `F1` y la matriz de confusión. El modelo final debe reducir errores importantes sin mostrar una brecha excesiva entre entrenamiento y validación.

In [3]:
from pathlib import Path
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

SOURCE_DIR = Path("datos_imagenes_originales")
DATA_DIR = Path("datos_clasificacion")
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("TensorFlow:", tf.__version__)
print("Fuente esperada:", SOURCE_DIR.resolve())

TensorFlow: 2.21.0
Fuente esperada: C:\Users\peral\OneDrive\Documentos\Universidad - EAFIT\Semestre V\Inteligencia Artificial\Tareas\ia-2026-2-actividades-JOANPERALTA\semana-11\datos_imagenes_originales


In [4]:
def collect_images(folder):
    return [path for path in folder.rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"No existe {SOURCE_DIR}. Crea una carpeta por clase dentro de esa ruta y agrega tus imágenes."
    )

class_dirs = sorted(path for path in SOURCE_DIR.iterdir() if path.is_dir())
if len(class_dirs) < 2:
    raise ValueError("Se necesitan al menos dos carpetas de clase dentro de datos_imagenes_originales.")

records = []
for class_dir in class_dirs:
    for image_path in collect_images(class_dir):
        records.append({"path": image_path, "label": class_dir.name})

images_df = pd.DataFrame(records)
if images_df.empty:
    raise ValueError("No se encontraron imágenes JPG, JPEG, PNG, BMP o WEBP.")

print(images_df.groupby("label").size().to_string())
print(f"Total de imágenes: {len(images_df)}")

FileNotFoundError: No existe datos_imagenes_originales. Crea una carpeta por clase dentro de esa ruta y agrega tus imágenes.

In [ ]:
# Se divide primero en train y un bloque temporal; luego ese bloque se separa en validation y test.
train_df, holdout_df = train_test_split(
    images_df,
    test_size=0.30,
    random_state=SEED,
    stratify=images_df["label"],
)
validation_df, test_df = train_test_split(
    holdout_df,
    test_size=0.50,
    random_state=SEED,
    stratify=holdout_df["label"],
)

if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)

for split_name, split_df in {
    "train": train_df,
    "validation": validation_df,
    "test": test_df,
}.items():
    for _, row in split_df.iterrows():
        destination_dir = DATA_DIR / split_name / row["label"]
        destination_dir.mkdir(parents=True, exist_ok=True)
        destination = destination_dir / row["path"].name
        shutil.copy2(row["path"], destination)

print({split: len(frame) for split, frame in {"train": train_df, "validation": validation_df, "test": test_df}.items()})
print("El test queda reservado y no se usa para ajustar hiperparámetros.")

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED
)
validation_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "validation", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    shuffle=False
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
train_ds = train_ds.prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

print("Clases:", class_names)
print("Número de clases:", len(class_names))

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.10),
], name="data_augmentation")

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    )
]

history_base = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=15,
    callbacks=callbacks,
)

history_base_df = pd.DataFrame(history_base.history)
ax = history_base_df[["accuracy", "val_accuracy"]].plot(figsize=(8, 4), title="Transfer learning con base congelada")
ax.set_xlabel("Época")
ax.set_ylabel("Accuracy")
plt.show()

## 2. Evaluación del primer modelo

Esta es la primera medición sobre test. No se modifica el modelo con estos resultados; se guardan para compararlos con el modelo ajustado.

In [ ]:
def collect_predictions(model, dataset):
    probabilities = model.predict(dataset, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    labels = np.concatenate([labels for _, labels in dataset], axis=0)
    return labels, predictions, probabilities

base_loss, base_accuracy = model.evaluate(test_ds, verbose=0)
base_labels, base_predictions, base_probabilities = collect_predictions(model, test_ds)

print(f"Pérdida en test: {base_loss:.4f}")
print(f"Accuracy en test: {base_accuracy:.4f}")
print(classification_report(base_labels, base_predictions, target_names=class_names, zero_division=0))

base_metrics = {
    "modelo": "Base congelada",
    "test_loss": base_loss,
    "test_accuracy": base_accuracy,
}

## 3. Fine-tuning

Se descongelan solo las últimas capas de la base MobileNetV2. La tasa de aprendizaje es mucho menor para evitar destruir las representaciones útiles aprendidas en ImageNet.

In [ ]:
base_model.trainable = True
fine_tune_from = max(0, len(base_model.layers) - 30)
for layer in base_model.layers[:fine_tune_from]:
    layer.trainable = False

# BatchNormalization suele permanecer congelada en datasets pequeños.
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(
    train_ds,
    validation_data=validation_ds,
    initial_epoch=len(history_base.history["loss"]),
    epochs=len(history_base.history["loss"]) + 10,
    callbacks=callbacks,
)

fine_history_df = pd.DataFrame(history_fine.history)
fine_history_df[["accuracy", "val_accuracy"]].plot(figsize=(8, 4), title="Fine-tuning")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.show()

In [ ]:
fine_loss, fine_accuracy = model.evaluate(test_ds, verbose=0)
fine_labels, fine_predictions, fine_probabilities = collect_predictions(model, test_ds)

print(f"Pérdida en test después de fine-tuning: {fine_loss:.4f}")
print(f"Accuracy en test después de fine-tuning: {fine_accuracy:.4f}")
print(classification_report(fine_labels, fine_predictions, target_names=class_names, zero_division=0))

comparison = pd.DataFrame([
    base_metrics,
    {"modelo": "Fine-tuning", "test_loss": fine_loss, "test_accuracy": fine_accuracy},
])
comparison["cambio_accuracy"] = comparison["test_accuracy"].diff()
comparison

In [ ]:
cm = confusion_matrix(fine_labels, fine_predictions)
fig, ax = plt.subplots(figsize=(7, 7))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False
)
ax.set_title("Matriz de confusión en test después de fine-tuning")
plt.tight_layout()
plt.show()

In [ ]:
test_paths = [path for path in (DATA_DIR / "test").rglob("*") if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS]
test_paths = sorted(test_paths)

errors = []
for path, true_index, predicted_index, probabilities in zip(
    test_paths, fine_labels, fine_predictions, fine_probabilities
):
    if true_index != predicted_index:
        errors.append({
            "imagen": str(path),
            "clase_real": class_names[true_index],
            "clase_predicha": class_names[predicted_index],
            "confianza": float(np.max(probabilities)),
        })

errors_df = pd.DataFrame(errors, columns=["imagen", "clase_real", "clase_predicha", "confianza"])
errors_df = errors_df.sort_values("confianza", ascending=False)
print(f"Errores encontrados: {len(errors_df)} de {len(test_paths)} imágenes")
display(errors_df.head(10))

# Muestra hasta cinco errores para apoyar el análisis cualitativo.
examples = errors_df.head(5)
if len(examples) > 0:
    fig, axes = plt.subplots(1, len(examples), figsize=(18, 4))
    for ax, (_, error) in zip(np.atleast_1d(axes), examples.iterrows()):
        image = tf.keras.utils.load_img(error["imagen"], target_size=IMAGE_SIZE)
        ax.imshow(image)
        ax.set_title(f"Real: {error['clase_real']}\nPred: {error['clase_predicha']} ({error['confianza']:.2f})")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No hubo errores en test; analiza cinco predicciones correctas difíciles o cercanas.")

## 4. Análisis de al menos cinco errores

Completa esta tabla después de observar las imágenes mostradas por la celda anterior. Si hubo menos de cinco errores, analiza también predicciones correctas con baja confianza.

| # | Imagen | Clase real | Clase predicha | ¿Qué pudo causar el error? | ¿Qué haría para mejorarlo? |
|---:|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |
| 4 | | | | | |
| 5 | | | | | |

Posibles causas para comprobar, sin asumirlas: fondo dominante, poca iluminación, oclusión, encuadre, resolución, clase visualmente parecida, desbalance o etiquetas incorrectas. La explicación debe referirse a la imagen y no solo a una causa genérica.

## 5. Conclusiones

Completa estas conclusiones con los valores que aparecen en `comparison`:

- El modelo con la base congelada obtuvo un accuracy de test de **[completar]**.
- Después del fine-tuning, el accuracy de test fue **[completar]**, un cambio de **[completar]** puntos.
- La clase con más confusiones fue **[completar]**, principalmente con **[completar]**.
- Los errores analizados muestran que **[completar patrón observado]**.
- La siguiente mejora prioritaria sería **[completar]**, porque **[justificación basada en los datos]**.

**Conclusión final:** el fine-tuning **[mejoró / no mejoró]** el modelo bajo esta partición. Esta conclusión se limita al dataset recolectado, sus condiciones de captura y la separación utilizada; no implica que el modelo generalice a cualquier imagen del mundo real.